In [1]:
# Rating 
# Price for two 
# Cuisine 
# Locality

In [2]:
import pandas as pd 
import requests 
import numpy as np
import time
import random

In [3]:
def get_restaurant_details_api(page_url_path):
    api_url = f"https://www.zomato.com/webroutes/getPage?page_url={page_url_path}" # This URL targets the internal API directly

    headers = {
        'accept': '*/*',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'x-zomato-csrft': 'e84022ea16c0db8abdcf4356ae701cf0',
        'referer': f'https://www.zomato.com{page_url_path}'
    } 
    #Bypassing bot detection using headers provided by cURL
    #Cookies to ensure the session is valid
    cookies = {
        'PHPSESSID': '558da14e8dea0af02aa3ed2109afbdfa',
        'csrf': 'e84022ea16c0db8abdcf4356ae701cf0',
        'fbcity': '26'
    }
    try:
        response = requests.get(api_url,headers=headers,cookies=cookies)
        if response.status_code == 200:
            data = response.json()
            page_data = data.get("page_data",{})
            sections = page_data.get("sections",{})

            page_info = data.get("page_info", {})
            page_title = page_info.get("pageTitle", "")
            
            # Logic to extract "Locality" from the title string
            try:
                locality_extracted = page_title.split(',')[1].strip()
            except (IndexError, AttributeError):
                locality_extracted = "Bhopal"

            basic_info = sections.get("SECTION_BASIC_INFO", {})
            res_details = sections.get("SECTION_RES_DETAILS", {})

            return {
                "name": basic_info.get("name"),
                "rating": basic_info.get("rating", {}).get("aggregate_rating"),
                "votes": basic_info.get("rating", {}).get("votes"),
                "cuisines": basic_info.get("cuisine_string"),
                "price": res_details.get("CFT_DETAILS", {}).get("cfts", [{}])[0].get("title"),
                "locality": locality_extracted
            }
    except Exception as e:
        print(f"Error parsing API for {page_url_path}: {e}")

    return None
                   

In [4]:
df_paths_ = pd.read_csv('bhopal_cleaned_paths.csv')
scraped_results = [] 
print(f"Starting API extraction for {len(df_paths_)} resturants")
for index, row in df_paths_.iterrows():
    path = row['api_path']
    
    print(f"[{index+1}/{len(df_paths_)}] Fetching data for: {path}")
    
    # Call the API function we built earlier
    data = get_restaurant_details_api(path)
    
    if data:
        scraped_results.append(data)
        print(f"   Successfully retrieved: {data['name']}")
    else:
        print(f"   Warning: Could not get data for {path}")

    # 4. CRITICAL: Add a random delay to avoid bot detection
    # This makes the script wait between 2 and 5 seconds before the next call
    time.sleep(random.uniform(2, 5))

# 5. Convert the list of dictionaries into a final DataFrame
df_final = pd.DataFrame(scraped_results)

# 6. Save the final results for your Machine Learning model
df_final.to_csv('bhopal_final_dataset.csv', index=False)

print("\nAll data has been harvested and saved to 'bhopal_final_dataset.csv'!")

Starting API extraction for 374 resturants
[1/374] Fetching data for: /bhopal/la-kuchina-jehan-numa-palace-tt-nagar/info
   Successfully retrieved: La Kuchina - Jehan Numa Palace
[2/374] Fetching data for: /bhopal/farzi-cafe-habib-ganj/info
   Successfully retrieved: Farzi Cafe
[3/374] Fetching data for: /bhopal/lil-china-pure-veg-pan-asian-kohefiza/info
   Successfully retrieved: Lil China- Pure Veg Pan Asian
[4/374] Fetching data for: /bhopal/pirates-of-grill-maharana-pratap-nagar/info
   Successfully retrieved: Pirates Of Grill
[5/374] Fetching data for: /bhopal/suruchis-24x7-gulmohar-colony/info
   Successfully retrieved: Suruchi's 24x7
[6/374] Fetching data for: /bhopal/sankalp-maharana-pratap-nagar/info
   Successfully retrieved: Sankalp
[7/374] Fetching data for: /bhopal/coriander-leaf-jehan-numa-retreat-tt-nagar/info
   Successfully retrieved: Coriander Leaf- Jehan Numa Retreat
[8/374] Fetching data for: /bhopal/kebabsville-sayaji-hotel-tt-nagar/info
   Successfully retrieved: 